In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import numpy as np

# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [2]:
class IrisDataset(Dataset):
    def __init__(self, X_data, y_data):
        # Convert data to PyTorch tensors of appropriate types
        self.X = torch.from_numpy(X_data).type(torch.FloatTensor)
        self.y = torch.from_numpy(y_data).type(torch.LongTensor) # CrossEntropyLoss expects LongTensor
        self.len = self.X.shape[0]

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.len

# Create dataset and dataloader instances
train_dataset = IrisDataset(X_train, y_train)
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)


In [3]:
class IrisModel(nn.Module):
    def __init__(self):
        super(IrisModel, self).__init__()
        self.fc1 = nn.Linear(4, 16)  # Input layer (4) to hidden layer (16)
        self.relu = nn.ReLU()      # ReLU activation function
        self.fc2 = nn.Linear(16, 3) # Hidden layer (16) to output layer (3)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x # The CrossEntropyLoss function will apply softmax


In [4]:
model = IrisModel()
criterion = nn.CrossEntropyLoss() # Combines nn.LogSoftmax() and nn.NLLLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01) # Adam optimizer is a good default
num_epochs = 100


In [5]:
model.train() # Set model to training mode

for epoch in range(num_epochs):
    for i, (inputs, labels) in enumerate(train_loader):
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad() # Zero the gradients
        loss.backward()       # Backpropagate the gradients
        optimizer.step()      # Update model parameters

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [10/100], Loss: 0.5679
Epoch [20/100], Loss: 0.3985
Epoch [30/100], Loss: 0.2658
Epoch [40/100], Loss: 0.1342
Epoch [50/100], Loss: 0.1000
Epoch [60/100], Loss: 0.1303
Epoch [70/100], Loss: 0.0965
Epoch [80/100], Loss: 0.0362
Epoch [90/100], Loss: 0.0158
Epoch [100/100], Loss: 0.0221


In [6]:
model.eval() # Set model to evaluation mode
with torch.no_grad(): # Disable gradient calculation for efficiency
    X_test_tensor = torch.from_numpy(X_test).type(torch.FloatTensor)
    y_test_tensor = torch.from_numpy(y_test).type(torch.LongTensor)

    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs.data, 1) # Get the class with the highest probability

    total = y_test_tensor.size(0)
    correct = (predicted == y_test_tensor).sum().item()
    accuracy = 100 * correct / total

    print(f'Accuracy of the model on the test set: {accuracy:.2f} %')


Accuracy of the model on the test set: 100.00 %
